# 01. Test Embedding Models

Compare different embedding models: MiniLM vs BGE for nutrition domain.


In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import time
from config.settings import EmbeddingConfig
from src.embeddings import EmbedderFactory, MiniLMEmbedder, BGEEmbedder
from sklearn.metrics.pairwise import cosine_similarity

/Users/soham/Desktop/nutribot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data


In [2]:
# Load nutrition documents
df = pd.read_csv('../data/processed/nutrition_docs.csv')
documents = df['content'].tolist()
print(f"Loaded {len(documents)} documents")

Loaded 10 documents


## Test Queries


In [3]:
test_queries = [
    "What are the benefits of protein?",
    "How much water should I drink daily?",
    "Best foods for weight loss?",
    "How to balance macronutrients?",
]

print(f"Testing with {len(test_queries)} queries")

Testing with 4 queries


## Test MiniLM Embedder


In [4]:
# Initialize MiniLM
print("Loading MiniLM embedder...")
minilm_start = time.time()
minilm = MiniLMEmbedder(device='cpu')
minilm_load_time = time.time() - minilm_start
print(f"MiniLM loaded in {minilm_load_time:.2f}s")

# Embed documents
print(f"Embedding {len(documents)} documents...")
minilm_start = time.time()
minilm_embeddings = minilm.embed_batch(documents, show_progress=True)
minilm_embed_time = time.time() - minilm_start
print(f"Documents embedded in {minilm_embed_time:.2f}s")
print(f"Embedding shape: {minilm_embeddings.shape}")

2026-06-15 20:17:19,921 - src.embeddings.minilm - INFO - Loading MiniLM embedder from sentence-transformers/all-MiniLM-L6-v2


Loading MiniLM embedder...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19838.05it/s]
2026-06-15 20:17:26,136 - src.embeddings.minilm - INFO - MiniLM embedder loaded successfully on cpu


MiniLM loaded in 6.22s
Embedding 10 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.73it/s]

Documents embedded in 0.06s
Embedding shape: (10, 384)


## Test BGE Embedder


In [5]:
# Initialize BGE
print("Loading BGE embedder...")
bge_start = time.time()
bge = BGEEmbedder(device='cpu')
bge_load_time = time.time() - bge_start
print(f"BGE loaded in {bge_load_time:.2f}s")

# Embed documents
print(f"Embedding {len(documents)} documents...")
bge_start = time.time()
bge_embeddings = bge.embed_batch(documents, show_progress=True)
bge_embed_time = time.time() - bge_start
print(f"Documents embedded in {bge_embed_time:.2f}s")
print(f"Embedding shape: {bge_embeddings.shape}")

2026-06-15 20:17:31,723 - src.embeddings.bge - INFO - Loading BGE embedder from BAAI/bge-large-en-v1.5


Loading BGE embedder...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12363.62it/s]
2026-06-15 20:17:48,712 - src.embeddings.bge - INFO - BGE embedder loaded successfully on cpu


BGE loaded in 16.99s
Embedding 10 documents...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.09s/it]

Documents embedded in 2.09s
Embedding shape: (10, 1024)


## Compare Query Results


In [6]:
# Test a query
test_query = test_queries[0]
print(f"Query: {test_query}\n")

# MiniLM results
minilm_query_emb = minilm.embed_query(test_query)
minilm_scores = cosine_similarity([minilm_query_emb], minilm_embeddings)[0]
minilm_top_k = np.argsort(minilm_scores)[::-1][:3]

print("MiniLM Top 3 Results:")
for i, idx in enumerate(minilm_top_k):
    print(f"  {i+1}. (score: {minilm_scores[idx]:.3f}) {documents[idx][:80]}...")

print()

# BGE results
bge_query_emb = bge.embed_query(test_query)
bge_scores = cosine_similarity([bge_query_emb], bge_embeddings)[0]
bge_top_k = np.argsort(bge_scores)[::-1][:3]

print("BGE Top 3 Results:")
for i, idx in enumerate(bge_top_k):
    print(f"  {i+1}. (score: {bge_scores[idx]:.3f}) {documents[idx][:80]}...")

Query: What are the benefits of protein?

MiniLM Top 3 Results:
  1. (score: 0.546) Protein is essential for muscle growth and repair. The recommended daily intake ...
  2. (score: 0.364) Balanced meals should include proteins, carbohydrates, and healthy fats. This co...
  3. (score: 0.360) Vitamins and minerals are essential micronutrients. Vitamin D supports bone heal...

BGE Top 3 Results:
  1. (score: 0.653) Protein is essential for muscle growth and repair. The recommended daily intake ...
  2. (score: 0.640) Balanced meals should include proteins, carbohydrates, and healthy fats. This co...
  3. (score: 0.565) Carbohydrates are the body's primary energy source. Complex carbs like whole gra...


## Performance Comparison


In [7]:
comparison = pd.DataFrame({
    'Model': ['MiniLM', 'BGE'],
    'Load Time (s)': [minilm_load_time, bge_load_time],
    'Embedding Time (s)': [minilm_embed_time, bge_embed_time],
    'Dimension': [minilm.dimension, bge.dimension],
    'Avg Score': [
        np.mean([minilm_scores[minilm_top_k].mean() for _ in test_queries]),
        np.mean([bge_scores[bge_top_k].mean() for _ in test_queries])
    ]
})

print("\nPerformance Comparison:")
print(comparison.to_string(index=False))
print(f"\nTime per document:")
print(f"  MiniLM: {minilm_embed_time/len(documents)*1000:.2f}ms/doc")
print(f"  BGE: {bge_embed_time/len(documents)*1000:.2f}ms/doc")


Performance Comparison:
 Model  Load Time (s)  Embedding Time (s)  Dimension  Avg Score
MiniLM       6.216294            0.060705        384   0.423394
   BGE      16.991993            2.092442       1024   0.619560

Time per document:
  MiniLM: 6.07ms/doc
  BGE: 209.24ms/doc
